# **STEP 2 — Initialize ChromaDB**

In [ ]:
!pip install chromadb
import chromadb

client = chromadb.Client()
collection = client.create_collection(name="rag_collection")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

# **STEP 3 — Load Embedding Model**

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# **STEP 4 — Sample Documents**

In [ ]:
documents = [
    "Artificial Intelligence is the simulation of human intelligence.",
    "Machine Learning is a subset of AI.",
    "Deep Learning uses neural networks.",
    "Natural Language Processing deals with text data.",
    "RAG combines retrieval and generation."
]

ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]

# **STEP 5 — Convert to Embeddings**

In [ ]:
embeddings = model.encode(documents)

# **STEP 6 — Store in ChromaDB**

In [ ]:
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=ids
)

# **STEP 7 — Similarity Search**

In [ ]:
query = "What is AI?"
query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)

print(results)

{'ids': [['doc1', 'doc2']], 'embeddings': None, 'documents': [['Artificial Intelligence is the simulation of human intelligence.', 'Machine Learning is a subset of AI.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[0.5604256391525269, 0.700352132320404]]}


# **STEP 8 — Display Results**

In [ ]:
for doc in results['documents'][0]:
    print(doc)

Artificial Intelligence is the simulation of human intelligence.
Machine Learning is a subset of AI.


# **STEP 9 — Retrieval Function**

In [ ]:
def retrieve(query):
    query_embedding = model.encode([query])
    results = collection.query(query_embeddings=query_embedding, n_results=2)
    return results['documents'][0]

print(retrieve("Explain machine learning"))

['Machine Learning is a subset of AI.', 'Deep Learning uses neural networks.']


# **STEP 10 — Understanding RAG Workflow**

Here's a breakdown of the Retrieval Augmented Generation (RAG) workflow:

*   **Step 1: User query** - The process begins with a natural language query from the user.
*   **Step 2: Convert query to embedding** - The user's query is converted into a numerical vector (embedding) using an embedding model. This allows for semantic comparison with stored documents.
*   **Step 3: Retrieve similar documents** - The query embedding is used to search a vector database (like ChromaDB) to find documents whose embeddings are most similar to the query's embedding. These are the most relevant documents.
*   **Step 4: Pass to LLM (next lab)** - The retrieved relevant documents, along with the original user query, are then passed to a Large Language Model (LLM) as context to generate a more informed and accurate response.